# Run Pipeline

This notebook follows the current local workflow:

1. Extract BTC archives into `data/keyframes/` and `data/map-keyframes/`
2. Import metadata into `data/index/metadata.jsonl`
3. Generate keyframe captions when transcripts are unavailable
4. Train LoRA CLIP by session, resuming between sessions
5. Extract CLIP features for the keyframes
6. Build the local two-level FAISS index
7. Run a local search test and display the results

In [ ]:
from pathlib import Path
import os

def _discover_project_root() -> Path:
    env_root = os.getenv('AIC_PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser()
        if candidate.exists():
            return candidate.resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'backend').is_dir():
            return candidate.resolve()

    raise RuntimeError('Set AIC_PROJECT_ROOT or open the notebook inside the project folder.')

PROJECT_ROOT = _discover_project_root()
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

## Step 0 - Check the runtime

Use this cell to confirm whether the notebook is attached to an NVIDIA GPU or running on CPU.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

## Step 1 - Extract BTC archives

Run this only once per dataset update. It creates the raw `keyframes/` and `map-keyframes/` folders that the rest of the pipeline reads.

In [ ]:
# Preview what will be extracted
subprocess.run([sys.executable, 'scripts/extract_btc_data.py', '--dry-run'], cwd=PROJECT_ROOT, check=True)

In [ ]:
# Extract for real
subprocess.run([sys.executable, 'scripts/extract_btc_data.py'], cwd=PROJECT_ROOT, check=True)

## Step 2 - Import metadata

This writes the canonical `data/index/metadata.jsonl` used by training, captioning, feature extraction, and indexing. Keep transcript disabled unless `data/videos/` contains source videos.

In [ ]:
# Metadata only. Add '--with-transcript' only when data/videos contains source videos.
subprocess.run([sys.executable, 'scripts/import_btc_data.py', '--force'], cwd=PROJECT_ROOT, check=True)

## Step 3 - Generate keyframe captions

Captions improve LoRA training when transcripts are unavailable. Run the smoke test first, then run the full caption step if the model fits your GPU/RAM.

In [ ]:
CAPTION_TEST_LIMIT = 100
CAPTION_BATCH_SIZE = 1
CAPTION_MODEL = 'Salesforce/blip-image-captioning-large'
CAPTION_MODEL_TYPE = 'blip'
CAPTION_PROMPT = 'aic'
CAPTION_MIN_WORDS = 8

print('Caption settings:')
print(f'  test_limit={CAPTION_TEST_LIMIT}')
print(f'  batch_size={CAPTION_BATCH_SIZE}')
print(f'  model={CAPTION_MODEL}')
print(f'  prompt={CAPTION_PROMPT}')
print(f'  min_words={CAPTION_MIN_WORDS}')

In [ ]:
# Caption smoke test
caption_test_cmd = [
    sys.executable,
    '-m', 'backend.preprocessing.generate_captions',
    '--limit', str(CAPTION_TEST_LIMIT),
    '--batch-size', str(CAPTION_BATCH_SIZE),
    '--model-name', CAPTION_MODEL,
    '--model-type', CAPTION_MODEL_TYPE,
    '--prompt', CAPTION_PROMPT,
    '--min-words', str(CAPTION_MIN_WORDS),
    '--force',
]
print('Running:', ' '.join(caption_test_cmd))
subprocess.run(caption_test_cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
# Full caption run. Skip this if you only want to train from existing captions/titles.
caption_full_cmd = [
    sys.executable,
    '-m', 'backend.preprocessing.generate_captions',
    '--batch-size', str(CAPTION_BATCH_SIZE),
    '--model-name', CAPTION_MODEL,
    '--model-type', CAPTION_MODEL_TYPE,
    '--prompt', CAPTION_PROMPT,
    '--min-words', str(CAPTION_MIN_WORDS),
    '--force',
]
print('Running:', ' '.join(caption_full_cmd))
subprocess.run(caption_full_cmd, cwd=PROJECT_ROOT, check=True)

## Step 4 - Train LoRA by session

Train in short sessions, then resume from the saved checkpoint. That keeps each run manageable and makes it easy to continue later.

Suggested pattern:
- Session 1: train on a smaller subset or fewer epochs
- Session 2+: resume from `data/index/lora_weights.pt`
- Increase `--limit` only when the previous session is stable

In [ ]:
import torch

TRAIN_LIMIT = 0
TEST_LIMIT = 100
TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 32
TRAIN_NUM_WORKERS = 0
TEST_NUM_WORKERS = 0
FEATURE_BATCH_SIZE = 16
FEATURE_TEST_LIMIT = 100
TRAIN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Training settings:')
print(f'  limit={TRAIN_LIMIT} (full train)')
print(f'  test_limit={TEST_LIMIT} (smoke test)')
print(f'  epochs={TRAIN_EPOCHS}')
print(f'  batch_size={TRAIN_BATCH_SIZE}')
print(f'  num_workers={TRAIN_NUM_WORKERS}')
print(f'  device={TRAIN_DEVICE}')
print(f'  feature_batch_size={FEATURE_BATCH_SIZE}')

In [ ]:
# Session 0: TEST
import subprocess, sys

train_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TEST_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TEST_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Session 1: fresh training run
# Increase --epochs if you want a longer first session.
import subprocess, sys

train_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TRAIN_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Session 2: resume from the saved checkpoint
# Run this after Session 1 if you want to continue training in another pass.
import subprocess, sys

resume_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TRAIN_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
    '--resume',
]
print('Running:', ' '.join(resume_cmd))
subprocess.run(resume_cmd, cwd=PROJECT_ROOT, check=True)


## Step 5 - Extract CLIP features

This reads your current keyframes and writes `.npy` vectors to `data/clip-features/`.

In [ ]:
# Smoke test 100 keyframes before the full feature extraction run.
feature_test_cmd = [
    sys.executable,
    'scripts/extract_clip_features.py',
    '--num-workers', str(TEST_NUM_WORKERS),
    '--batch-size', str(FEATURE_BATCH_SIZE),
    '--limit', str(FEATURE_TEST_LIMIT),
]
print('Running:', ' '.join(feature_test_cmd))
subprocess.run(feature_test_cmd, cwd=PROJECT_ROOT, check=True)

### Step 5.1 - Full feature extraction
Run this after the 100-keyframe smoke test passes and the final LoRA checkpoint is ready.

In [ ]:
feature_cmd = [
    sys.executable,
    'scripts/extract_clip_features.py',
    '--num-workers', str(TEST_NUM_WORKERS),
    '--batch-size', str(FEATURE_BATCH_SIZE),
]
print('Running:', ' '.join(feature_cmd))
subprocess.run(feature_cmd, cwd=PROJECT_ROOT, check=True)

## Step 6 - Build the local FAISS index

This creates `data/index/video.index` for keyframe lookup and `data/index/scene.index` for coarse filtering.

In [ ]:
subprocess.run([sys.executable, '-m', 'backend.embedding.build_index'], cwd=PROJECT_ROOT, check=True)

## Step 7 - Local search test

This is the notebook-style test flow similar to the old version: encode a query, search the local FAISS index, and print the top hits.

In [ ]:
import json
from pathlib import Path

import faiss
import numpy as np

from backend.config import FAISS_INDEX_PATH, FAISS_METADATA_PATH
from backend.embedding.clip_encoder import encode_text

In [ ]:
index = faiss.read_index(str(FAISS_INDEX_PATH))
with open(FAISS_METADATA_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Loaded {index.ntotal} vectors and {len(metadata)} metadata rows.')

In [ ]:
query = 'a photo of a tree'
vec = encode_text(query).reshape(1, -1).astype(np.float32)
faiss.normalize_L2(vec)

top_k = 10
scores, indices = index.search(vec, top_k)

print(f"Query: {query}")
for i, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    if idx < 0 or idx >= len(metadata):
        continue
    item = metadata[idx]
    print(f"#{i:02d} | score={score:.4f} | {item.get('video_id', '')} | frame={item.get('frame_id', '')} | pts={item.get('pts_time', 0.0):.2f}s")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

n = min(top_k, 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f"Search results: {query}")

for i, ax in enumerate(axes.flat):
    if i >= n:
        ax.axis('off')
        continue
    idx = indices[0][i]
    if idx < 0 or idx >= len(metadata):
        ax.axis('off')
        continue

    item = metadata[idx]
    img_path = Path(item.get('path', ''))
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert('RGB'))
    else:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=ax.transAxes)

    ax.set_title(f"#{i+1} | {item.get('video_id', '')} | {item.get('frame_id', '')}")
    ax.axis('off')

plt.tight_layout()
plt.show()